# ERA5 Data Extraction via Google Earth Engine
This notebook pulls ERA5 data from GEE directly into an `xarray` dataset and saves it as a `.nc` file to Google Drive and your local folder.

In [ ]:
!pip install earthengine-api wxee geopandas

### Setup Google Drive Mount (For Colab)

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_path = '/content/drive/MyDrive/RainCast_Data/'
    print("Mounted Google Drive")
except ImportError:
    print("Not running in Colab. Using local directory.")
    drive_path = './Data/'

os.makedirs(drive_path, exist_ok=True)

### Authenticate with Google Earth Engine
Make sure your Google Cloud Project has the Earth Engine API enabled.

In [ ]:
import ee
import wxee

try:
    # Replace 'your-project' with your actual GCP project ID
    ee.Initialize(project='your-project')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='your-project')

# Initialize wxee for xarray integration
wxee.Initialize()

### Define Region of Interest (Indore)
We use a bounding box around Indore. Later, we will use a precise shapefile to clip this data during training.

In [ ]:
# Indore Bounding Box (Longitude Min, Latitude Min, Longitude Max, Latitude Max)
roi = ee.Geometry.Rectangle([75.5, 22.5, 76.2, 23.0])

### Fetch ERA5 Data
We download Daily Total Precipitation and Maximum Temperature.

In [ ]:
era5 = ee.ImageCollection("ECMWF/ERA5/DAILY") \
        .select(['total_precipitation', 'maximum_2m_air_temperature']) \
        .filterDate('1979-01-01', '2025-01-01')

# Download SRTM Digital Elevation Model (Topography)
dem = ee.Image("USGS/SRTMGL1_003").select('elevation')

print("Downloading ERA5 data to xarray... This may take a while depending on the date range.")
# Convert GEE ImageCollection to xarray Dataset (scale is in meters, ~11km for ERA5)
ds = era5.wx.to_xarray(region=roi, scale=11132)

# Convert DEM to xarray and add it to our main dataset
dem_ds = dem.wx.to_xarray(region=roi, scale=11132)
ds['elevation'] = dem_ds['elevation']
print(ds)

### Save as NetCDF (.nc)

In [ ]:
nc_path = os.path.join(drive_path, 'indore_era5.nc')
ds.to_netcdf(nc_path)
print(f"Successfully saved NetCDF to {nc_path}")